# 📘 Week 1 · Day 5-7: 시각화 & EDA 전략

## 🎯 학습 목표
- Matplotlib / Seaborn 으로 **데이터가 말하는 것**을 읽는다
- Kaggle에서 쓰이는 **EDA 패턴 10가지**를 익힌다
- 타깃 변수와 피처의 관계를 수치·그래프로 파악한다
- **누설(Leakage) 의심 신호**를 EDA에서 미리 잡는다

> 💡 EDA(Exploratory Data Analysis, 탐색적 데이터 분석)는 모델보다 **더 중요**합니다. EDA로 발견한 인사이트 하나가 리더보드 100등 차이를 만듭니다.

---

## 1. 시각화 기본 셋업

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Kaggle 표준 스타일
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('Set2')

print("셋업 완료")

In [ ]:
# 실습용 Titanic 샘플 데이터 재생성
np.random.seed(42)
n = 891
titanic = pd.DataFrame({
    'PassengerId': range(1, n+1),
    'Survived':    np.random.binomial(1, 0.38, n),
    'Pclass':      np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
    'Sex':         np.random.choice(['male', 'female'], n, p=[0.65, 0.35]),
    'Age':         np.random.normal(30, 14, n).clip(0.5, 80),
    'SibSp':       np.random.poisson(0.5, n),
    'Parch':       np.random.poisson(0.38, n),
    'Fare':        np.random.exponential(32, n).round(2),
    'Embarked':    np.random.choice(['S', 'C', 'Q'], n, p=[0.72, 0.19, 0.09])
})
# 생존률과 상관 있는 패턴 의도적 주입
titanic.loc[titanic['Sex'] == 'female', 'Survived'] = np.random.binomial(1, 0.74, (titanic['Sex'] == 'female').sum())
titanic.loc[titanic['Sex'] == 'male', 'Survived'] = np.random.binomial(1, 0.19, (titanic['Sex'] == 'male').sum())
titanic.loc[np.random.choice(n, 177, replace=False), 'Age'] = np.nan
titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1
titanic.head()

---

## 2. EDA 10단계 체크리스트

Kaggle 프로가 EDA를 시작할 때 **반드시** 하는 10가지입니다.

### Step 1: 형태 & 타입 파악

In [ ]:
print("Shape:", titanic.shape)
print()
print("Dtypes:")
print(titanic.dtypes)
print()
print("Memory (MB):", titanic.memory_usage(deep=True).sum() / 1024**2)

### Step 2: 결측치 패턴 시각화

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# 결측치 비율 bar
missing_pct = titanic.isna().mean().sort_values(ascending=False)
missing_pct[missing_pct > 0].plot(kind='barh', ax=ax[0], color='salmon')
ax[0].set_title('Missing Value Ratio')
ax[0].set_xlabel('missing %')

# 결측치 히트맵 (missingno 대체)
sns.heatmap(titanic.isna(), cbar=False, yticklabels=False, ax=ax[1], cmap='gray_r')
ax[1].set_title('Missing Value Pattern')

plt.tight_layout()
plt.show()

### Step 3: 타깃 변수 분포 (가장 먼저!)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

titanic['Survived'].value_counts().plot(kind='bar', ax=ax[0], color=['#e74c3c','#2ecc71'])
ax[0].set_title('Survived Count')
ax[0].set_xticklabels(['Dead(0)', 'Alive(1)'], rotation=0)

titanic['Survived'].value_counts(normalize=True).plot(kind='pie', ax=ax[1],
                                                       autopct='%.1f%%',
                                                       colors=['#e74c3c','#2ecc71'])
ax[1].set_title('Survived Ratio')
ax[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("클래스 불균형도:", titanic['Survived'].value_counts(normalize=True).round(3).to_dict())

> 💡 **클래스 불균형**(예: 95:5)이면 Accuracy 대신 **F1, AUC, Precision-Recall** 을 봐야 하고, SMOTE/class_weight 를 써야 합니다.

### Step 4: 수치형 피처 분포 (히스토그램 + KDE)

In [ ]:
num_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(num_cols):
    ax = axes[i//3, i%3]
    titanic[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{col} distribution')
    ax.set_xlabel(col)
axes[1, 2].axis('off')
plt.tight_layout()
plt.show()

> 💡 **왜도(skewness)가 심한** Fare 같은 변수는 `np.log1p` 로 변환하면 모델 성능이 오를 때가 많습니다.

In [ ]:
# 왜도 체크
from scipy.stats import skew
for col in ['Age', 'Fare']:
    print(f"{col:5s} skew = {skew(titanic[col].dropna()):.2f}")

In [ ]:
# log 변환 전/후 비교
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
titanic['Fare'].hist(bins=40, ax=ax[0], color='tomato')
ax[0].set_title(f'Fare (skew={skew(titanic["Fare"]):.2f})')

np.log1p(titanic['Fare']).hist(bins=40, ax=ax[1], color='seagreen')
ax[1].set_title(f'log1p(Fare) (skew={skew(np.log1p(titanic["Fare"])):.2f})')

plt.tight_layout()
plt.show()

### Step 5: 범주형 피처 분포

In [ ]:
cat_cols = ['Pclass', 'Sex', 'Embarked']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(cat_cols):
    titanic[col].value_counts().plot(kind='bar', ax=axes[i], color='coral')
    axes[i].set_title(f'{col}')
    axes[i].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### Step 6: 타깃 vs 피처 (가장 중요한 단계)

In [ ]:
# 범주형 → 그룹별 타깃 평균
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(cat_cols):
    titanic.groupby(col)['Survived'].mean().plot(kind='bar', ax=axes[i], color='mediumseagreen')
    axes[i].set_title(f'Survival rate by {col}')
    axes[i].axhline(titanic['Survived'].mean(), ls='--', c='red', label='overall')
    axes[i].legend()
    axes[i].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 수치형 → 타깃별 분포 비교 (violin/boxplot)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.violinplot(data=titanic, x='Survived', y='Age', ax=axes[0])
axes[0].set_title('Age by Survived')

sns.boxplot(data=titanic, x='Survived', y='Fare', ax=axes[1])
axes[1].set_title('Fare by Survived')
axes[1].set_ylim(0, 200)  # outlier 잘라내기
plt.tight_layout()
plt.show()

### Step 7: 피처 간 상관관계

In [ ]:
# 수치형 상관계수 heatmap
num_df = titanic.select_dtypes(include='number')
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# 타깃과 상관 top
print("\nSurvived 상관 TOP:")
print(corr['Survived'].drop('Survived').abs().sort_values(ascending=False).round(3))

> ⚠️ **주의**: 상관계수는 **선형 관계**만 잡습니다. 비선형(예: U자형) 관계는 못 잡으니 scatter plot도 함께 봐야 합니다.

### Step 8: 이변량/다변량 관계

In [ ]:
# pairplot - 한 번에 다 보기
sns.pairplot(titanic[['Age', 'Fare', 'FamilySize', 'Survived']].dropna(),
             hue='Survived', diag_kind='kde', height=2)
plt.suptitle('Pairplot', y=1.02)
plt.show()

In [ ]:
# 3변수 교차 분석 (pivot + heatmap)
pivot = titanic.pivot_table(index='Pclass', columns='Sex', values='Survived', aggfunc='mean')
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax)
ax.set_title('Survival rate: Pclass × Sex')
plt.tight_layout()
plt.show()

### Step 9: 이상치 탐지

In [ ]:
# Boxplot으로 outlier 시각화
fig, ax = plt.subplots(figsize=(12, 4))
titanic[['Age', 'Fare', 'SibSp', 'Parch']].plot(kind='box', ax=ax)
ax.set_title('Outlier check (boxplot)')
plt.tight_layout()
plt.show()

In [ ]:
# IQR 기반 outlier 개수
def count_outliers_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return ((s < lower) | (s > upper)).sum()

for col in ['Age', 'Fare', 'SibSp', 'Parch']:
    print(f"{col:5s}: outlier {count_outliers_iqr(titanic[col]):4d}개")

### Step 10: Train/Test 분포 비교 (누수 방지 & 검증 전략)

실제 대회에서는 `train.csv` 와 `test.csv` 의 **피처 분포가 동일해야** 합니다. 다르면 모델이 일반화되지 않습니다.

In [ ]:
# 예시: test set이 train보다 평균 연령이 높다면?
train_age = titanic['Age'].dropna()
test_age  = titanic['Age'].dropna() + np.random.normal(5, 2, len(titanic['Age'].dropna()))  # 인위적으로 shift

fig, ax = plt.subplots(figsize=(10, 4))
sns.kdeplot(train_age, label='train', ax=ax, fill=True)
sns.kdeplot(test_age,  label='test',  ax=ax, fill=True)
ax.set_title('Train vs Test: Age distribution (예시)')
ax.legend()
plt.tight_layout()
plt.show()

# KS test로 분포 차이 수치화
from scipy.stats import ks_2samp
stat, p = ks_2samp(train_age, test_age)
print(f"KS statistic = {stat:.4f}, p-value = {p:.4f}")
print("→ p < 0.05 이면 분포가 다르다고 판단")

> 🚨 **Adversarial Validation**: train과 test를 구분하는 분류기를 만들었을 때 AUC가 0.5 근처면 분포 동일, 1에 가까우면 분포 다름. 다음 주에 배웁니다.

---

## 3. Kaggle 시각화 필살 패턴

In [ ]:
# 패턴 1: 타깃 vs 피처 3종 한 번에 (Kaggle 노트북 단골 스타일)
def plot_target_vs_feature(df, feature, target='Survived'):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 분포
    df[feature].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].set_title(f'{feature} distribution')

    # 타깃별 분포
    for t in df[target].unique():
        df[df[target]==t][feature].hist(bins=30, alpha=0.5, ax=axes[1], label=f'{target}={t}')
    axes[1].legend()
    axes[1].set_title(f'{feature} by {target}')

    # 구간별 타깃 평균
    df['_bin'] = pd.qcut(df[feature], q=10, duplicates='drop')
    df.groupby('_bin', observed=False)[target].mean().plot(kind='bar', ax=axes[2], color='orange')
    axes[2].set_title(f'{target} rate by {feature} quantile')
    axes[2].tick_params(axis='x', rotation=45)
    df.drop(columns='_bin', inplace=True)

    plt.tight_layout()
    plt.show()

plot_target_vs_feature(titanic.dropna(subset=['Age']), 'Age')

In [ ]:
plot_target_vs_feature(titanic, 'Fare')

---

## 4. EDA에서 발견해야 할 신호

| 신호 | 의미 | 대응 |
|------|------|------|
| 한 피처 결측 > 50% | 정보 없음 가능 | 드롭 고려, 결측 자체가 정보면 flag 추가 |
| 타깃과 상관 1.0 가까움 | **Data Leakage** 의심 | 반드시 피처 생성 로직 점검 |
| Train/Test 분포 큰 차이 | 시간/샘플링 편향 | Adversarial Validation으로 확인 |
| 극단적 outlier | 측정 오류 가능 | winsorize, log 변환, 제거 |
| 범주 수 > 1000 | 고차원 카디널리티 | Target/Frequency Encoding |
| 타깃 불균형 (99:1) | | class_weight, SMOTE, focal loss |

---

## 📝 Day 5-7 체크리스트
- [ ] 결측치 패턴 시각화 가능
- [ ] 타깃 분포 / 클래스 불균형 확인 습관
- [ ] 범주형 × 타깃 교차 분석
- [ ] 상관계수 heatmap 해석
- [ ] Train/Test 분포 비교 개념 이해

다음 주는 **머신러닝의 수학적 기초**와 **선형/로지스틱 회귀**를 실전 코드와 함께 배웁니다.